In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Preprocessing

In [52]:
df = pd.read_csv('../data/raw/Product Data.csv', sep=';', encoding='utf-8-sig')

col_rename = {
    "No_": "SKU",
    "Nazwa_PL": "Name_PL",
    "Opis_PL": "Description_PL",
    "Skład_PL": "Composition_PL",
    "Nazwa_EN": "Name_EN",
    "Opis_EN": "Description_EN",
    "Skład_EN": "Composition_EN",
    "Gender": "Gender",
    "Gramatura": "Grammage",
    "Net Weight": "Weight_net",
    "PVS Item Type Code$5452f323-059e-499a-9753-5d2c07eef904": "Item_Type_Code",
    "PVS Item Quality Code$5452f323-059e-499a-9753-5d2c07eef904": "Item_Quality_Code",
    "PVS Quality Type Code$5452f323-059e-499a-9753-5d2c07eef904": "Quality_Type_Code"
}
df = df.rename(columns=col_rename)


## Feature engeneering

In [53]:
df_proc = df.copy()

#Drop Category columns
df_proc.drop(['Quality_Type_Code', 'Item_Type_Code'], axis=1, inplace=True)

#Construct descriptive categories
df_proc['Item_Quality_Code'] = df_proc['Item_Quality_Code'].replace({'APD': 'API'})
quality_code_desc = {
    'TOE': 'Bags_-_Backpacks',
    'TOP': 'Bags_-_Travel, weekend bags',
    'TOW': 'Bags_-_Suitcases',
    'TOS': 'Bags_-_Sports bags',
    'TOA': 'Bags_-_Paper, gift, Christmas bags',
    'TOI': 'Bags_-_Isothermal bags',
    'TOL': 'Bags_-_Laptop bags',
    'TOC': 'Bags_-_Belt bags / fanny packs',
    'TOK': 'Bags_-_Conference / document bags',
    'TOZ': 'Bags_-_Beach bags',
    'TOZB': 'Bags_-_Cotton / shopping bags',
    'TOG': 'Bags_-_Drawstring bags',
    'TOH': 'Bags_-_Shoulder bags',
    'TOJ': 'Bags_-_Jute bags',
    'TOB': 'Bags_-_Phone bags',
    'TOF': 'Bags_-_Felt bags',
    'TOY': 'Bags_-_Cosmetic bags',
    'TOM': 'Bags_-_Sets',
    'TON': 'Bags_-_Other',
    'CZD': 'Caps_-_Baseball caps',
    'CZZ': 'Caps_-_Winter caps',
    'CZA': 'Caps_-_Kids caps',
    'CZK': 'Caps_-_Hats',
    'CZW': 'Caps_-_Military caps',
    'CZS': 'Caps_-_Sports caps',
    'CZV': 'Caps_-_Visors',
    'CZY': 'Caps_-_Flat caps',
    'CZM': 'Caps_-_Morphs / balaclavas',
    'CZI': 'Caps_-_Other',
    'PAK': 'Umbrellas_-_Classic',
    'PAG': 'Umbrellas_-_Golf',
    'PAS': 'Umbrellas_-_Folding',
    'KBC': 'Mugs, bottles, thermos_-_Ceramic and porcelain',
    'KBZ': 'Mugs, bottles, thermos_-_Glass',
    'KBS': 'Mugs, bottles, thermos_-_Car mugs',
    'KBF': 'Mugs, bottles, thermos_-_Cups',
    'KBA': 'Mugs, bottles, thermos_-_Sports bottles',
    'KBB': 'Mugs, bottles, thermos_-_Thermos flasks',
    'KBI': 'Mugs, bottles, thermos_-_Isothermal',
    'KBX': 'Mugs, bottles, thermos_-_Sets',
    'KBN': 'Mugs, bottles, thermos_-_Other',
    'BRK': 'Small items_-_Keychains',
    'BRA': 'Small items_-_Stress relievers',
    'BRZ': 'Small items_-_Lighters',
    'BRL': 'Small items_-_Mirrors',
    'BRT': 'Small items_-_Travel accessories',
    'BRN': 'Small items_-_Fan accessories',
    'BRD': 'Small items_-_Brand items',
    'BRB': 'Small items_-_Lanyards',
    'BRI': 'Small items_-_Other',
    'APG': 'Writing instruments_-_Plastic pens',
    'APF': 'Writing instruments_-_Metal pens',
    'APH': 'Writing instruments_-_Multi-function pens',
    'API': 'Writing instruments_-_Other pens',
    'APK': 'Writing instruments_-_Ballpoint pens',
    'APP': 'Writing instruments_-_Fountain pens',
    'APO': 'Writing instruments_-_Pencils',
    'APA': 'Writing instruments_-_Crayons / colored pencils',
    'APC': 'Writing instruments_-_Highlighters',
    'APE': 'Writing instruments_-_Cases / pouches',
    'APZ': 'Writing instruments_-_Writing sets',
    'ABK': 'Office_-_Calendars',
    'ABA': 'Office_-_Calculators',
    'ABB': 'Office_-_Office accessories',
    'ABS': 'Office_-_Sticky notes',
    'ABE': 'Office_-_Business card / credit card holders',
    'ABN': 'Office_-_Notebooks',
    'ABO': 'Office_-_Wallets',
    'ABT': 'Office_-_Portfolios',
    'ABU': 'Office_-_Desk clocks',
    'ABR': 'Office_-_Wristwatches',
    'ABG': 'Office_-_Wall clocks',
    'ABI': 'Office_-_Other',
    'ABP': 'Office_-_Containers, organizers, stands',
    'ABD': 'Office_-_Alarm clocks',
    'ACY': 'Technology/USB_-_Mouse pads',
    'ACM': 'Technology/USB_-_Mice',
    'ACU': 'Technology/USB_-_USB drives',
    'ACL': 'Technology/USB_-_Laptop accessories',
    'ACT': 'Technology/USB_-_Phone accessories',
    'ACC': 'Technology/USB_-_Power banks',
    'ACB': 'Technology/USB_-_Chargers',
    'ACE': 'Technology/USB_-_Phone and laptop cases',
    'ACG': 'Technology/USB_-_Speakers and headphones',
    'ACI': 'Technology/USB_-_Other',
    'AKD': 'Kitchen_-_Boards and mats',
    'AKN': 'Kitchen_-_Carafes and glasses',
    'AKO': 'Kitchen_-_Openers',
    'AKU': 'Kitchen_-_Kitchen accessories',
    'AKZ': 'Kitchen_-_Spice sets / grinders',
    'AKP': 'Kitchen_-_Food containers',
    'AKW': 'Kitchen_-_Wine sets and accessories',
    'AKS': 'Kitchen_-_Cutlery / knives',
    'AKI': 'Kitchen_-_Other',
    'NAJ': 'Tools & Auto_-_Car accessories',
    'NAL': 'Tools & Auto_-_Flashlights / torches',
    'NAN': 'Tools & Auto_-_Knives / pocket knives',
    'NAT': 'Tools & Auto_-_Tape measures',
    'NAZ': 'Tools & Auto_-_Tool sets',
    'NAW': 'Tools & Auto_-_Multi-tools',
    'NAI': 'Tools & Auto_-_Other',
    'NAM': 'Tools & Auto_-_Machines and equipment',
    'FKM': 'Kids_-_Plush toys',
    'FKZ': 'Kids_-_Toys',
    'FKG': 'Kids_-_Rubber ducks',
    'FKA': 'Kids_-_School supplies',
    'FKI': 'Kids_-_Other',
    'RSY': 'Sport & leisure_-_Travel accessories',
    'RSN': 'Sport & leisure_-_Other',
    'RSI': 'Sport & leisure_-_Fitness accessories',
    'RSP': 'Sport & leisure_-_Beach items',
    'RSR': 'Sport & leisure_-_Cycling accessories',
    'RSH': 'Sport & leisure_-_BBQ and picnic',
    'RSJ': 'Sport & leisure_-_Balls',
    'RSO': 'Sport & leisure_-_Sunglasses',
    'RSG': 'Sport & leisure_-_Games and puzzles',
    'RSQ': 'Sport & leisure_-_Pedometers and activity monitors',
    'ADZ': 'Home & interior_-_Bathroom accessories',
    'ADI': 'Home & interior_-_Candles and fragrance sets',
    'ADL': 'Home & interior_-_Lamps',
    'ADG': 'Home & interior_-_Weather stations',
    'ADP': 'Home & interior_-_Boxes',
    'ADD': 'Home & interior_-_Photo frames',
    'ADN': 'Home & interior_-_Other',
    'ZBA': 'Health & safety_-_First aid kits',
    'ZBO': 'Health & safety_-_Reflective items',
    'ZBM': 'Health & safety_-_Personal care products',
    'ZBI': 'Health & safety_-_Other',
    'NBT': 'Awards & Jewellery_-_Trophies',
    'NBN': 'Awards & Jewellery_-_Awards and gift sets',
    'NBA': 'Awards & Jewellery_-_Plaques',
    'NBD': 'Awards & Jewellery_-_Diplomas',
    'NBB': 'Awards & Jewellery_-_Jewellery',
    'NBI': 'Awards & Jewellery_-_Other',
    'PZP': 'Pins & badges_-_Pins',
    'PZM': 'Pins & badges_-_Magnets',
    'PZZ': 'Pins & badges_-_Badges',
    'PZI': 'Pins & badges_-_Name badges / lanyards',
    'SSG': 'Seasonal & food_-_Food items',
    'SSO': 'Seasonal & food_-_Decorations and gifts',
    'AMF': 'Promotional items_-_Flags',
    'AMK': 'Promotional items_-_Boxes / cartons',
    'AMR': 'Promotional items_-_Advertising banners',
    'AMW': 'Promotional items_-_Bags and pouches',
    'AMI': 'Promotional items_-_Other',
    'MEW': 'Labels_-_Woven labels',
    'MEZ': 'Labels_-_Hang tags',
    'MEN': 'Labels_-_Patches',
    'MES': 'Labels_-_Sashes',
    'MET': 'Labels_-_Tapes',
    'MEJ': 'Labels_-_Die-cut labels',
    'MEE': 'Labels_-_Cardboard labels',
    'DUMMY': 'DUMMY_-_DUMMY',
    'TSK': 'T-shirt_-_Short sleeve',
    'TSP': 'T-shirt_-_Top',
    'TSD': 'T-shirt_-_Long sleeve',
    'TST': 'T-shirt_-_Tunics',
    'KPK': 'Polo_-_Short sleeve',
    'KPD': 'Polo_-_Long sleeve',
    'KSD': 'Shirts_-_Long sleeve',
    'KSK': 'Shirts_-_Short sleeve',
    'KSS': 'Shirts_-_3/4 sleeve',
    'POZ': 'Fleece_-_Full zip',
    'POK': 'Fleece_-_Quarter zip (1/4)',
    'BLZ': 'Sweatshirts_-_Classic (Sweatshirt)',
    'BLR': 'Sweatshirts_-_Zip-up',
    'BLK': 'Sweatshirts_-_Hooded (Hoodie)',
    'BLH': 'Sweatshirts_-_Zip-up hoodie',
    'KUE': 'Outerwear_-_Ponchos / Raincoats + sets',
    'KUW': 'Outerwear_-_Windbreakers',
    'KUP': 'Outerwear_-_Transitional jackets',
    'KUL': 'Outerwear_-_Fleece jackets',
    'KUZ': 'Outerwear_-_Winter jackets',
    'KUM': 'Outerwear_-_Blazers',
    'KUG': 'Outerwear_-_Jackets, suit jackets, ponchos',
    'KUC': 'Outerwear_-_Coats',
    'SF': 'Softshell_-_Softshell',
    'BEL': 'Gilets_-_Fleece',
    'BEU': 'Gilets_-_Padded',
    'BEP': 'Gilets_-_Transitional',
    'BES': 'Gilets_-_Softshell',
    'SDK': 'Trousers & Skirts_-_Shorts',
    'SDD': 'Trousers & Skirts_-_Trousers',
    'SDS': 'Trousers & Skirts_-_3/4 trousers',
    'SDU': 'Trousers & Skirts_-_Dresses',
    'SDP': 'Trousers & Skirts_-_Skirts',
    'SWZ': 'Knitwear_-_Pullovers',
    'SWR': 'Knitwear_-_Cardigans',
    'SWK': 'Knitwear_-_Waistcoats',
    'SWG': 'Knitwear_-_Turtlenecks',
    'AZS': 'Winter accessories_-_Fleece scarves',
    'AZR': 'Winter accessories_-_Gloves',
    'AZZ': 'Winter accessories_-_Winter sets',
    'AZN': 'Winter accessories_-_Ear muffs',
    'AZO': 'Winter accessories_-_Headbands',
    'BIF': 'Socks & Underwear_-_Briefs / Shorts / Thongs',
    'BIB': 'Socks & Underwear_-_Boxers / Briefs',
    'BIS': 'Socks & Underwear_-_Socks',
    'BID': 'Socks & Underwear_-_Bodysuits',
    'BIT': 'Socks & Underwear_-_Thermal underwear',
    'BIL': 'Socks & Underwear_-_Bibs',
    'GLK': 'Ties & Accessories_-_Ties / Bow ties',
    'GLS': 'Ties & Accessories_-_Scarves, shawls, bandanas',
    'GLP': 'Ties & Accessories_-_Belts',
    'OZM': 'Ties & Accessories_-_Blazers',
    'OMG': 'Ties & Accessories_-_Jackets, suit jackets, ponchos',
    'OZE': 'Ties & Accessories_-_Dress trousers',
    'OZO': 'Ties & Accessories_-_Shirts',
    'OZD': 'Ties & Accessories_-_Skirts',
    'OZA': 'Ties & Accessories_-_Waistcoats',
    'ODN': 'Kids clothing_-_Baby / infant',
    'ODD': 'Kids clothing_-_Children and youth',
    'OTF': 'Technical clothing_-_Aprons',
    'OTK': 'Technical clothing_-_Overalls / coveralls',
    'OTO': 'Technical clothing_-_High-visibility clothing',
    'OTA': 'Technical clothing_-_Accessories (knee pads, armbands, bandanas, etc.)',
    'OTR': 'Technical clothing_-_Cycling',
    'OTU': 'Technical clothing_-_Uniforms',
    'OTM': 'Technical clothing_-_Masks',
    'OBS': 'Footwear_-_Sports shoes',
    'OBL': 'Footwear_-_Summer (flip flops, sandals)',
    'OBK': 'Footwear_-_Rubber boots / wellies',
    'OBR': 'Footwear_-_Work shoes',
    'TDR': 'Home textiles_-_Towels',
    'TDK': 'Home textiles_-_Blankets',
    'TDS': 'Home textiles_-_Bathrobes',
}
df_proc['Item_Quality_Code_desc'] =df_proc['Item_Quality_Code'].map(quality_code_desc)

In [55]:
#Fill Gender Gap
df_proc['Gender'] = df_proc['Gender'].fillna('U')
df_proc['Gender'] = df_proc['Gender'].replace({'U': 'Unisex', 'W': 'Woman', 'M': 'Men', 'Y': 'Children'})

#Delete rows with no descriptive data
nan_cols = ['Name_PL', 'Description_PL', 'Name_EN', 'Description_EN']
mask_nan = (df_proc[nan_cols].fillna('').apply(lambda x: x.str.strip()).eq('').all(axis=1))
df_proc = df_proc[~mask_nan]

#Construct Descriptive category form SKU
mapping = {'T': 'Textiles', 'G': 'Gadgets'}
df_proc['Tree'] = df['SKU'].str[0].map(mapping)

#Fill 0 values in numeric columns
df_proc['Weight_net'] = df_proc['Weight_net'].str.replace(',', '.').pipe(pd.to_numeric, errors='coerce')
mask = df_proc['Tree'] == 'Textiles'
cols = ['Grammage', 'Weight_net']

for col in cols:
    median_val = df_proc.loc[mask, col].replace(0, np.nan).median()
    df_proc.loc[mask, col] = df_proc.loc[mask, col].replace(0, median_val)

df_proc

,SKU,Name_PL,Composition_PL,Description_PL,Name_EN,Composition_EN,Description_EN,Gender,Grammage,Weight_net,Item_Quality_Code,Item_Quality_Code_desc,Tree
0,GAB/00311/L335,NaN,NaN,NaN,Long White Cooking Apron with Front Pocket,100% Cotton/Cotton twill,"Long white cooking apron with front pocket, 60...",Unisex,180,0.160,OTF,Technical clothing_-_Aprons,Gadgets
1,GAB/00512/01,Torba na Zakupy z Krótkimi Uszami,100% Bawełna,"Długość uszu: 35 x 2,5 cm.",Shopping Bag with Short Handles,100% Cotton,"Handles: 35 x 2,5 cm.",Unisex,140,0.069,TOZB,Bags_-_Cotton / shopping bags,Gadgets
2,GAB/00512/02,Torba na Zakupy z Krótkimi Uszami,100% Bawełna,"Długość uszu: 35 x 2,5 cm.",Shopping Bag with Short Handles,100% Cotton,"Handles: 35 x 2,5 cm.",Unisex,140,0.069,TOZB,Bags_-_Cotton / shopping bags,Gadgets
3,GAB/00512/03,Torba na Zakupy z Krótkimi Uszami,100% Bawełna,"Długość uszu: 35 x 2,5 cm.",Shopping Bag with Short Handles,100% Cotton,"Handles: 35 x 2,5 cm.",Unisex,140,0.069,TOZB,Bags_-_Cotton / shopping bags,Gadgets
4,GAB/00512/04,Torba na Zakupy z Krótkimi Uszami,100% Bawełna,"Długość uszu: 35 x 2,5 cm.",Shopping Bag with Short Handles,100% Cotton,"Handles: 35 x 2,5 cm.",Unisex,140,0.069,TOZB,Bags_-_Cotton / shopping bags,Gadgets
...,...,...,...,...,...,...,...,...,...,...,...,...,...
294125,TYO/YK910/N399/4,NaN,NaN,NaN,Hi-Vis Top Cool Super Light V-Neck T-Shirt,100% Polyester,Top cool V-neck mesh high visibility T-shirt. ...,Unisex,130,0.199,OTO,Technical clothing_-_High-visibility clothing,Textiles
294126,TYO/YK910/N399/5,NaN,NaN,NaN,Hi-Vis Top Cool Super Light V-Neck T-Shirt,100% Polyester,Top cool V-neck mesh high visibility T-shirt. ...,Unisex,130,0.205,OTO,Technical clothing_-_High-visibility clothing,Textiles
294127,TYO/YK910/N399/6,NaN,NaN,NaN,Hi-Vis Top Cool Super Light V-Neck T-Shirt,100% Polyester,Top cool V-neck mesh high visibility T-shirt. ...,Unisex,130,0.213,OTO,Technical clothing_-_High-visibility clothing,Textiles
294128,TYO/YK910/N399/7,NaN,NaN,NaN,Hi-Vis Top Cool Super Light V-Neck T-Shirt,100% Polyester,Top cool V-neck mesh high visibility T-shirt. ...,Unisex,130,0.218,OTO,Technical clothing_-_High-visibility clothing,Textiles


In [56]:
df_clean = df_proc
df_clean_en = df_proc.drop(['Name_PL', 'Composition_PL', 'Description_PL'], axis=1)
df_clean.to_csv('../data/processed/Product Data - Clean.csv', index=False, encoding='utf-8-sig', sep=';')
df_clean_en.to_csv('../data/processed/Product Data - CleanEn.csv', index=False, encoding='utf-8-sig', sep=';')